# range
`range` is not just a way to count. In real systems, it helps model bounded retries, index-based scans, partition walks, chunk processing, and fixed iteration budgets. It becomes dangerous when engineers use it without thinking about bounds, exclusivity, and intent.

## 1. Problem First
Why does this matter in real systems?
- Retry loops often need exact attempt counts.
- Batch processing may walk data in fixed-size chunks.
- Off-by-one errors in `range` silently skip or overrun expected work.

In [1]:
for attempt in range(3):
    print(attempt)

0
1
2


## 2. Minimal Concept
Core syntax:
- `range(stop)`
- `range(start, stop)`
- `range(start, stop, step)`
- The `stop` value is excluded.

## 3. Mental Model
How Python actually behaves
`range` does not build a full list in Python 3. It creates a compact range object that can produce numbers on demand. That means it is memory-efficient even for large spans, but its semantics still depend on start, stop, step, and exclusivity.

In [2]:
values = range(1, 6)
print(values)
print(list(values))
print(4 in values)

range(1, 6)
[1, 2, 3, 4, 5]
True


## 4. Internal Mechanics
`range` stores boundaries and step, not every integer. It supports iteration, membership checks, and indexing without materializing a list unless you ask for one.

In [3]:
import dis

def count_down():
    for value in range(3, 0, -1):
        print(value)

dis.dis(count_down)
print(range(0, 10, 2)[2])

  3           0 RESUME                   0

  4           2 LOAD_GLOBAL              1 (NULL + range)
             12 LOAD_CONST               1 (3)
             14 LOAD_CONST               2 (0)
             16 LOAD_CONST               3 (-1)
             18 CALL                     3
             26 GET_ITER
        >>   28 FOR_ITER                13 (to 58)
             32 STORE_FAST               0 (value)

  5          34 LOAD_GLOBAL              3 (NULL + print)
             44 LOAD_FAST                0 (value)
             46 CALL                     1
             54 POP_TOP
             56 JUMP_BACKWARD           15 (to 28)

  4     >>   58 END_FOR
             60 RETURN_CONST             0 (None)
4


## 5. Edge Cases
Where it breaks:
- `stop` is excluded, which causes off-by-one errors.
- A wrong step sign can produce no values at all.
- Using `range(len(items))` when direct iteration is clearer often adds unnecessary indexing.

In [4]:
print(list(range(5)))
print(list(range(5, 0)))
print(list(range(5, 0, -1)))

[0, 1, 2, 3, 4]
[]
[5, 4, 3, 2, 1]


## 6. Performance Thinking
- Creating a `range` object is O(1) memory.
- Iterating a `range(n)` is O(n).
- `range` is usually better than building a list of numbers just to loop over it.
- The bigger performance issue is usually what happens inside the loop.

## 7. Real Use Case
A batch job may process records in fixed chunk sizes by stepping through indexes.

In [5]:
records = list(range(10))
chunk_size = 3
for start in range(0, len(records), chunk_size):
    chunk = records[start:start + chunk_size]
    print(chunk)

[0, 1, 2]
[3, 4, 5]
[6, 7, 8]
[9]


## 8. Anti-Pattern
What beginners do wrong:
- Write `range(len(items))` when they only need the values.
- Forget that `stop` is excluded.
- Use magic numeric bounds without naming what they represent.

In [6]:
items = ["a", "b", "c"]
for index in range(len(items)):
    print(items[index])

for item in items:
    print(item)

a
b
c
a
b
c


## 9. Interview Signals
Questions FAANG asks:
- Why is `range` memory-efficient in Python 3?
- What is the most common off-by-one mistake with `range`?
- When is `range(len(sequence))` appropriate?
- How would you use `range` for chunking or bounded retries?

## 10. Exercise (Non-trivial)
Design a chunked processor for a large CSV import. Walk the input in bounded slices, handle a final partial chunk correctly, and avoid off-by-one bugs in both chunk boundaries and retry counters.

In [7]:
def chunk_records(records, chunk_size):
    chunks = []
    for start in range(0, len(records), chunk_size):
        chunks.append(records[start:start + chunk_size])
    return chunks

# Task:
# 1. Validate chunk_size.
# 2. Explain why the final chunk still works.
# 3. Show where off-by-one mistakes could appear.